# 01 · RSCD exploratory data analysis (PRD C1)

Class counts, a sample grid per friction class, image statistics and split sanity checks.

Set the `RSCD_ROOT` environment variable to the dataset folder (on Colab:
`%env RSCD_ROOT=/content/drive/MyDrive/RSCD`). Without it the notebook builds a small
**mock** dataset so every cell still runs; mock numbers mean nothing.

Section 4 reads the split files written by `python scripts/prepare_rscd.py --root ...`.

In [ ]:
import json
import os
import random
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from PIL import Image

from roadcond.config import load_config, resolve_path
from roadcond.data.mock import make_mock_rscd
from roadcond.data.rscd import LEAK_REPORT, SPLITS, index_split, load_splits, prepare
from roadcond.metrics import format_table

cfg = load_config()
rscd = cfg["rscd"]
CLASSES = rscd["classes"]
ROOT = Path(os.environ.get("RSCD_ROOT", resolve_path(rscd["root"])))
SPLITS_DIR = resolve_path(cfg["paths"]["splits"])
MOCK = not ROOT.is_dir()
if MOCK:
    ROOT = resolve_path("data/raw/rscd_mock")
    SPLITS_DIR = resolve_path("data/processed/rscd_mock_splits")
    if not ROOT.is_dir():
        make_mock_rscd(ROOT)
    if not (SPLITS_DIR / LEAK_REPORT).is_file():
        prepare(cfg, ROOT, SPLITS_DIR, train_size=120)
    print("RSCD not found: using MOCK data. The numbers below mean nothing.")
print("RSCD root:", ROOT)
print("Split files:", SPLITS_DIR)

In [ ]:
records = {s: index_split(ROOT, s, rscd) for s in SPLITS}
{s: len(r) for s, r in records.items()}

## 1. Class counts

Friction-class counts per official split, then the full 27-class labels in train.
The training subset is balanced later; val and test keep their official distribution.

In [ ]:
counts = {s: Counter(r.friction for r in records[s]) for s in SPLITS}
print(format_table(["class", *SPLITS], [[c, *(counts[s][c] for s in SPLITS)] for c in CLASSES]))

fig, axes = plt.subplots(1, 3, figsize=(15, 3.5))
for ax, s in zip(axes, SPLITS):
    ax.bar(CLASSES, [counts[s][c] for c in CLASSES])
    ax.set_title(f"{s} ({len(records[s]):,} images)")
    ax.tick_params(axis="x", rotation=45)
plt.tight_layout()

In [ ]:
labels = Counter(r.label for r in records["train"]).most_common()
fig, ax = plt.subplots(figsize=(8, 0.25 * len(labels) + 1))
ax.barh([name for name, _ in labels][::-1], [n for _, n in labels][::-1])
ax.set_title("train: full RSCD labels")
plt.tight_layout()

## 2. Sample grid per friction class

In [ ]:
rng = random.Random(0)
N = 8
by_class = {c: [r for r in records["train"] if r.friction == c] for c in CLASSES}
fig, axes = plt.subplots(len(CLASSES), N, figsize=(2 * N, 1.6 * len(CLASSES)))
for row, c in zip(axes, CLASSES):
    for ax in row:
        ax.axis("off")
    for ax, r in zip(row, rng.sample(by_class[c], min(N, len(by_class[c])))):
        with Image.open(ROOT / r.path) as img:
            ax.imshow(img.convert("RGB"))
    row[0].set_title(c, loc="left", fontsize=10)
plt.tight_layout()

## 3. Image statistics

On a sample of up to 200 train images per class: image sizes (PIL reports width x height,
which checks the PRD's "240 x 360"), and mean brightness and saturation per class.
Big gaps between classes here are colour cues the histogram baseline can use; they can
also be shortcuts (lighting, time of day) rather than road surface.

In [ ]:
SAMPLE = 200
stats = []
for c in CLASSES:
    for r in rng.sample(by_class[c], min(SAMPLE, len(by_class[c]))):
        with Image.open(ROOT / r.path) as img:
            hsv = np.asarray(img.convert("HSV"), dtype=np.float32) / 255
            stats.append({"class": c, "size": img.size, "mode": img.mode,
                          "saturation": float(hsv[..., 1].mean()),
                          "brightness": float(hsv[..., 2].mean())})
print("Sizes (width, height):", Counter(s["size"] for s in stats).most_common(5))
print("Modes:", Counter(s["mode"] for s in stats).most_common())

fig, axes = plt.subplots(1, 2, figsize=(13, 3.5))
for ax, key in zip(axes, ("brightness", "saturation")):
    ax.boxplot([[s[key] for s in stats if s["class"] == c] for c in CLASSES])
    ax.set_xticks(range(1, len(CLASSES) + 1), CLASSES, rotation=45)
    ax.set_title(f"mean {key} per image")
plt.tight_layout()

## 4. Split sanity checks

1. No file appears in two official splits (by path or by file name).
2. The saved train subset only uses official train images and is balanced.
3. The saved val and test lists equal the official splits exactly (untouched).
4. The leak report: train images dropped as near-duplicates of val/test images.

In [ ]:
paths = {s: {r.path for r in records[s]} for s in SPLITS}
names = {s: {Path(p).name for p in paths[s]} for s in SPLITS}
for a, b in [("train", "val"), ("train", "test"), ("val", "test")]:
    print(f"{a}/{b}: shared paths {len(paths[a] & paths[b])}, "
          f"shared file names {len(names[a] & names[b])}")

In [ ]:
saved = load_splits(SPLITS_DIR)
subset = {r.path for r in saved["train"]}
assert subset <= paths["train"], "subset has files outside the official train split"
assert not subset & (paths["val"] | paths["test"]), "subset overlaps val/test"
for s in ("val", "test"):
    assert [r.path for r in saved[s]] == [r.path for r in records[s]], f"{s} list differs"
print("Subset per class:", dict(Counter(r.friction for r in saved["train"])))
print("Saved val/test lists match the official splits.")

report = json.loads((SPLITS_DIR / LEAK_REPORT).read_text())
skip = ("examples", "full_labels", "split_sizes", "subset_per_class")
print(json.dumps({k: v for k, v in report.items() if k not in skip}, indent=2))

Flagged pairs, closest first. If different scenes appear here, `max_hamming` is too loose;
if obvious repeats slip through (check consecutive file names), it is too tight or the file
names carry a drive id worth setting as `rscd.dedup.drive_id_regex`.

In [ ]:
examples = report["examples"][:6]
if examples:
    fig, axes = plt.subplots(len(examples), 2, figsize=(7, 2.2 * len(examples)), squeeze=False)
    for (left, right), ex in zip(axes, examples):
        for ax, path, title in ((left, ex["train"], "train"),
                                (right, ex["match"], f"{ex['match_split']}, {ex['distance']} bits")):
            with Image.open(ROOT / path) as img:
                ax.imshow(img.convert("RGB"))
            ax.set_title(f"{title}: {Path(path).name}", fontsize=8)
            ax.axis("off")
    plt.tight_layout()
else:
    print("No near-duplicates found among the checked train images.")